In [53]:
import re
import pandas as pd
%pip install tqdm

#%pip install presidio-analyzer presidio-anonymizer


Note: you may need to restart the kernel to use updated packages.


In [58]:
import pandas as pd
import re
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider
from tqdm import tqdm

# ==============================================================================
# 1. SETUP INICIAL E LEITURA DO ARQUIVO (COMO JÁ FIZEMOS)
# ==============================================================================
caminho_do_arquivo = '../../data/conversa_teste.txt'
try:
    with open(caminho_do_arquivo, 'r', encoding='utf-8') as file:
        linhas = file.readlines()
except FileNotFoundError:
    print(f"Erro: Arquivo não encontrado no caminho: {caminho_do_arquivo}")
    linhas = []

dados = []
padrao = re.compile(r'\[(\d{2}/\d{2}/\d{2}), (\d{2}:\d{2}:\d{2})\] ([^:]+): (.*)')

if linhas:
    for linha in linhas:
        if linha.strip():
            match = padrao.match(linha)
            if match:
                data, hora, interlocutor, mensagem = match.groups()
                dados.append({
                    'data': data,
                    'hora': hora,
                    'interlocutor': interlocutor.strip(),
                    'mensagem': mensagem.strip()
                })

df = pd.DataFrame(dados)

# Criando a coluna 'nome_fantasia'
mapa_nomes_fantasia = {nome: f'interlocutor_{i+1}' for i, nome in enumerate(interlocutores_unicos)}
df['nome_fantasia'] = df['interlocutor'].map(mapa_nomes_fantasia)

# Criando o DataFrame final formatado
df_final = df.reset_index().rename(columns={
    'index': 'test_id',
    'mensagem': 'message'
})
colunas_desejadas = ['test_id', 'data', 'hora', 'nome_fantasia', 'message']
df_final = df_final[colunas_desejadas]

print("DataFrame inicial pronto para anonimização:")
print(df_final.head())
print("-" * 50)


# ==============================================================================
# 2. SETUP DO PRESIDIO (COMO CORRIGIMOS)
# ==============================================================================
nlp_config = {
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "pt", "model_name": "pt_core_news_lg"}]
}
provider = NlpEngineProvider(nlp_configuration=nlp_config)
nlp_engine = provider.create_engine()
analyzer = AnalyzerEngine(nlp_engine=nlp_engine, supported_languages=["pt"])
anonymizer = AnonymizerEngine()

# ==============================================================================
# 3. APLICAÇÃO DA ANONIMIZAÇÃO NO DATAFRAME
# ==============================================================================
# Registrando o 'tqdm' para funcionar com o pandas.apply
tqdm.pandas(desc="Anonimizando mensagens")

# Função que anonimiza um único texto
def anonimizar_texto(texto):
    if not isinstance(texto, str) or not texto:
        return "" # Retorna vazio se o texto não for uma string ou estiver vazio
    try:
        resultados = analyzer.analyze(text=texto, language='pt')
        texto_anonimizado = anonymizer.anonymize(text=texto, analyzer_results=resultados)
        return texto_anonimizado.text
    except Exception as e:
        # Em caso de erro inesperado em uma linha, retorna o texto original
        print(f"Erro ao anonimizar o texto: '{texto[:50]}...'. Erro: {e}")
        return texto

# Aplicando a função na coluna 'message' com uma barra de progresso
df_final['message_anonymized'] = df_final['message'].progress_apply(anonimizar_texto)

df_final = df_final.drop(columns=['message']).rename(columns={'message_anonymized': 'message'})
# ==============================================================================
# 4. EXIBIÇÃO DO RESULTADO FINAL
# ==============================================================================
print("\nDataFrame final com a coluna de mensagens anonimizadas:")
df_final.head()
df_final.to_csv('../../data/conversa_teste_anonimizada.csv', index=False)

DataFrame inicial pronto para anonimização:
   test_id      data      hora   nome_fantasia  \
0        0  17/06/25  10:44:14  interlocutor_1   
1        1  17/06/25  10:44:14  interlocutor_2   
2        2  17/06/25  10:44:15  interlocutor_1   
3        3  17/06/25  10:44:21  interlocutor_2   
4        4  17/06/25  10:44:28  interlocutor_2   

                                             message  
0  ‎Messages and calls are end-to-end encrypted. ...  
1        ‎CADU Russo created group “BGX <> Greenlab”  
2                              ‎CADU Russo added you  
3                               Oi pessoal, bom dia!  
4               Finalizamos o onboarding da Greenlab  
--------------------------------------------------


Anonimizando mensagens: 100%|██████████| 4847/4847 [00:10<00:00, 447.29it/s]


DataFrame final com a coluna de mensagens anonimizadas:


In [59]:

df_final.head()

,test_id,data,hora,nome_fantasia,message
0,0,17/06/25,10:44:14,interlocutor_1,‎Messages and calls are end-to-end encrypted. ...
1,1,17/06/25,10:44:14,interlocutor_2,<ORGANIZATION> “<ORGANIZATION> <> Greenlab”
2,2,17/06/25,10:44:15,interlocutor_1,<ORGANIZATION>
3,3,17/06/25,10:44:21,interlocutor_2,"Oi pessoal, bom dia!"
4,4,17/06/25,10:44:28,interlocutor_2,Finalizamos o <LOCATION>


In [60]:
amostra = df_final.iloc[1000:1300]
amostra.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 1000 to 1299
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   test_id        300 non-null    int64 
 1   data           300 non-null    object
 2   hora           300 non-null    object
 3   nome_fantasia  300 non-null    object
 4   message        300 non-null    object
dtypes: int64(1), object(4)
memory usage: 11.8+ KB


In [61]:
amostra.to_csv('../../data/amostra_teste_anonimizada.csv', index=False)
amostra.head()

,test_id,data,hora,nome_fantasia,message
1000,1000,06/08/25,11:10:41,interlocutor_4,130k
1001,1001,06/08/25,11:10:46,interlocutor_4,cotação pfvr?
1002,1002,06/08/25,11:11:06,interlocutor_7,4930
1003,1003,06/08/25,11:11:39,interlocutor_7,<LOCATION>
1004,1004,06/08/25,11:12:12,interlocutor_4,fecha
